# Install required medical imaging and deep learning libraries

In [4]:
!pip install -q pydicom SimpleITK pylidc torch h5py pandas matplotlib tqdm

In [5]:
import os
import sys
import glob
import json
import logging
from pathlib import Path
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
import pydicom
import torch
from torch.utils.data import Dataset, DataLoader
import h5py
from tqdm.notebook import tqdm

import numpy as np
np.int = int
np.float = float
np.bool = bool
import pandas as pd

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully!")
else:
    print("Running in local / non-Colab environment.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mounted Google Drive successfully!


# Set dataset paths

In [6]:
if IN_COLAB:
    RAW_DATA_DIR = Path("/content/drive/MyDrive/Lung_Nodule_Project/raw_data")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Lung_Nodule_Project/processed_patches")
else:
    RAW_DATA_DIR = Path("./data/raw_data")
    OUTPUT_DIR = Path("./data/processed_patches")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATCHES_DIR = OUTPUT_DIR / "patches"
PATCHES_DIR.mkdir(parents=True, exist_ok=True)

# Configure ~/.pylidcrc to enable pylidc scan queries

In [7]:
config_path = Path.home() / ".pylidcrc"
config_content = f"""[pylidc]
path = {RAW_DATA_DIR.resolve()}
warn = False
"""
config_path.write_text(config_content)
print(f"Configured ~/.pylidcrc with dataset path: {RAW_DATA_DIR}")

Configured ~/.pylidcrc with dataset path: /content/drive/MyDrive/Lung_Nodule_Project/raw_data


# Import pylidc after configuration

In [8]:
import pylidc as pl
print(f"pylidc successfully imported!")

pylidc successfully imported!


#Preprocessing Pipeline Functions

In [9]:
def find_dicom_series_dir(scan_folder: Path) -> Optional[Path]:
    """Find the directory with the primary CT series (largest number of slices)."""
    candidate_dirs = []
    for root, _, files in os.walk(scan_folder):
        dcm_count = sum(1 for f in files if f.lower().endswith(".dcm") or not "." in f)
        if dcm_count > 10:
            candidate_dirs.append((Path(root), dcm_count))
    if not candidate_dirs:
        return None
    candidate_dirs.sort(key=lambda x: x[1], reverse=True)
    return candidate_dirs[0][0]

def load_dicom_volume_sitk(dicom_dir: Path) -> sitk.Image:
    """Load DICOM series into a 3D SimpleITK image with spatial metadata."""
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(dicom_dir))
    if not series_ids:
        file_names = reader.GetGDCMSeriesFileNames(str(dicom_dir))
    else:
        file_names = reader.GetGDCMSeriesFileNames(str(dicom_dir), series_ids[0])

    if not file_names:
        raise FileNotFoundError(f"No DICOM files found in: {dicom_dir}")

    reader.SetFileNames(file_names)
    image = reader.Execute()
    return image

def resample_volume_isotropic(
    image: sitk.Image,
    target_spacing: Tuple[float, float, float] = (1.0, 1.0, 1.0),
    default_value: float = -1000.0
) -> sitk.Image:
    """Resample 3D SimpleITK image to isotropic (1.0, 1.0, 1.0) mm spacing."""
    orig_spacing = np.array(image.GetSpacing(), dtype=np.float64)
    orig_size = np.array(image.GetSize(), dtype=np.int64)
    target_spacing = np.array(target_spacing, dtype=np.float64)

    new_size = np.round(orig_size * orig_spacing / target_spacing).astype(np.int64)

    resample = sitk.ResampleImageFilter()
    resample.SetInterpolator(sitk.sitkLinear)
    resample.SetOutputSpacing(target_spacing.tolist())
    resample.SetSize(new_size.tolist())
    resample.SetOutputDirection(image.GetDirection())
    resample.SetOutputOrigin(image.GetOrigin())
    resample.SetDefaultPixelValue(default_value)
    resample.SetOutputPixelType(sitk.sitkFloat32)

    return resample.Execute(image)

def apply_lung_window_and_normalize(
    volume_np: np.ndarray,
    min_hu: float = -1000.0,
    max_hu: float = 400.0
) -> np.ndarray:
    """Clip CT intensities to lung window [-1000, 400] HU and normalize to [0, 1]."""
    clipped = np.clip(volume_np, min_hu, max_hu).astype(np.float32)
    normalized = (clipped - min_hu) / (max_hu - min_hu)
    return normalized

def extract_3d_patch(
    volume_np: np.ndarray,
    centroid_zyx: Tuple[int, int, int],
    patch_size: Tuple[int, int, int] = (64, 64, 64),
    pad_value: float = 0.0
) -> Tuple[np.ndarray, bool]:
    """Extract a 3D subvolume patch (64x64x64) with zero-padding at boundaries."""
    cz, cy, cx = centroid_zyx
    pd, ph, pw = patch_size
    vz, vy, vx = volume_np.shape

    half_d, half_h, half_w = pd // 2, ph // 2, pw // 2
    z_min_req, z_max_req = cz - half_d, cz + (pd - half_d)
    y_min_req, y_max_req = cy - half_h, cy + (ph - half_h)
    x_min_req, x_max_req = cx - half_w, cx + (pw - half_w)

    z_min_vol, z_max_vol = max(0, z_min_req), min(vz, z_max_req)
    y_min_vol, y_max_vol = max(0, y_min_req), min(vy, y_max_req)
    x_min_vol, x_max_vol = max(0, x_min_req), min(vx, x_max_req)

    z_min_patch = z_min_vol - z_min_req
    z_max_patch = z_min_patch + (z_max_vol - z_min_vol)
    y_min_patch = y_min_vol - y_min_req
    y_max_patch = y_min_patch + (y_max_vol - y_min_vol)
    x_min_patch = x_min_vol - x_min_req
    x_max_patch = x_min_patch + (x_max_vol - x_min_vol)

    patch = np.full(patch_size, fill_value=pad_value, dtype=np.float32)
    is_padded = (z_min_req < 0 or z_max_req > vz or y_min_req < 0 or y_max_req > vy or x_min_req < 0 or x_max_req > vx)

    if (z_max_vol > z_min_vol) and (y_max_vol > y_min_vol) and (x_max_vol > x_min_vol):
        patch[z_min_patch:z_max_patch, y_min_patch:y_max_patch, x_min_patch:x_max_patch] = \
            volume_np[z_min_vol:z_max_vol, y_min_vol:y_max_vol, x_min_vol:x_max_vol]

    return patch, is_padded

print("Core preprocessing functions defined successfully!")

Core preprocessing functions defined successfully!


#Annotation Parsing

In [10]:
def parse_scan_nodules(scan: pl.Scan, resampled_sitk_img: sitk.Image, original_sitk_img: sitk.Image) -> List[Dict[str, Any]]:
    """Extract nodule clusters, consensus malignancy ratings, and resampled voxel coordinates."""
    nodules = []
    try:
        clusters = scan.cluster_annotations()
    except Exception as e:
        print(f"Annotation clustering error for {scan.patient_id}: {e}")
        return []

    for c_idx, cluster in enumerate(clusters):
        if not cluster:
            continue

        malignancies = [ann.malignancy for ann in cluster if ann.malignancy is not None]
        if not malignancies:
            continue
        consensus_mal = float(np.mean(malignancies))

        if consensus_mal < 3.0:
            mal_class = 0
        elif consensus_mal > 3.0:
            mal_class = 1
        else:
            mal_class = -1

        world_pts = []
        for ann in cluster:
            ann_vox = ann.centroid
            sitk_idx = (float(ann_vox[1]), float(ann_vox[0]), float(ann_vox[2]))
            try:
                world_pt = original_sitk_img.TransformContinuousIndexToPhysicalPoint(sitk_idx)
                world_pts.append(world_pt)
            except Exception:
                pass

        if not world_pts:
            continue

        avg_world_pt = np.mean(world_pts, axis=0)
        res_cont_idx = resampled_sitk_img.TransformPhysicalPointToContinuousIndex(tuple(avg_world_pt))
        res_voxel_zyx = (
            int(np.round(res_cont_idx[2])),
            int(np.round(res_cont_idx[1])),
            int(np.round(res_cont_idx[0]))
        )

        nodule_entry = {
            "nodule_id": f"{scan.patient_id}_nodule_{c_idx:03d}",
            "patient_id": scan.patient_id,
            "num_annotations": len(cluster),
            "consensus_malignancy": consensus_mal,
            "malignancy_class": mal_class,
            "subtlety": float(np.mean([ann.subtlety for ann in cluster if ann.subtlety])),
            "sphericity": float(np.mean([ann.sphericity for ann in cluster if ann.sphericity])),
            "margin": float(np.mean([ann.margin for ann in cluster if ann.margin])),
            "spiculation": float(np.mean([ann.spiculation for ann in cluster if ann.spiculation])),
            "texture": float(np.mean([ann.texture for ann in cluster if ann.texture])),
            "centroid_world_xyz": tuple(avg_world_pt),
            "centroid_voxel_zyx": res_voxel_zyx,
        }
        nodules.append(nodule_entry)

    return nodules

print("Annotation parser defined successfully!")

Annotation parser defined successfully!


#Batch Preprocessing Execution

In [11]:
import os
import shutil
from pathlib import Path
import pylidc as pl

# 1. Explicitly define paths
RAW_DATA_DIR = Path("/content/drive/MyDrive/Lung_Nodule_Project/raw_data")
RAW_DATA_DIR_STR = str(RAW_DATA_DIR)

print(f"Scanning directory: {RAW_DATA_DIR_STR}")
print("Restructuring downloaded folders to match LIDC-IDRI standards...")

# 2. Query pylidc for all scan associations
scans = pl.query(pl.Scan).all()
moved_count = 0

for scan in scans:
    # The 1.3.6... series folder created by the cloud downloader
    uid_folder = os.path.join(RAW_DATA_DIR_STR, scan.series_instance_uid)

    # The LIDC-IDRI-XXXX patient folder expected by pylidc
    patient_folder = os.path.join(RAW_DATA_DIR_STR, scan.patient_id)

    # If the raw Series UID folder exists in the root, move it inside the patient folder
    if os.path.exists(uid_folder) and os.path.isdir(uid_folder):
        os.makedirs(patient_folder, exist_ok=True)

        target_path = os.path.join(patient_folder, scan.series_instance_uid)
        shutil.move(uid_folder, target_path)
        moved_count += 1

print(f"\n✅ Reorganized {moved_count} CT series into their respective LIDC-IDRI patient folders!")

Scanning directory: /content/drive/MyDrive/Lung_Nodule_Project/raw_data
Restructuring downloaded folders to match LIDC-IDRI standards...

✅ Reorganized 4 CT series into their respective LIDC-IDRI patient folders!


In [13]:
def preprocess_full_dataset(raw_dir, out_dir, max_scans=None):
    import pylidc as pl
    import torch
    import SimpleITK as sitk
    from tqdm.notebook import tqdm
    import pandas as pd

    manifest_data = []
    scans = pl.query(pl.Scan).all()

    if max_scans is not None:
        scans = scans[:max_scans]

    PATCHES_DIR = out_dir / "patches"
    PATCHES_DIR.mkdir(parents=True, exist_ok=True)

    for scan in tqdm(scans, desc="Preprocessing Patients"):
        patient_id = scan.patient_id
        patient_folder = raw_dir / patient_id

        if not patient_folder.exists():
            continue

        dicom_dir = find_dicom_series_dir(patient_folder)
        if not dicom_dir:
            continue

        try:
            # Load and process 3D CT volume
            original_sitk_img = load_dicom_volume_sitk(dicom_dir)
            resampled_sitk_img = resample_volume_isotropic(original_sitk_img)

            vol_np = sitk.GetArrayFromImage(resampled_sitk_img)
            vol_np = apply_lung_window_and_normalize(vol_np)

            # Extract nodule metadata and coordinates
            nodules = parse_scan_nodules(scan, resampled_sitk_img, original_sitk_img)

            # Crop and save 3D patches
            for nodule in nodules:
                centroid = nodule["centroid_voxel_zyx"]
                patch_np, is_padded = extract_3d_patch(vol_np, centroid)

                # Add channel dimension: (1, 64, 64, 64)
                patch_tensor = torch.tensor(patch_np).unsqueeze(0)

                patch_path = PATCHES_DIR / f"{nodule['nodule_id']}.pt"
                torch.save({'tensor': patch_tensor}, patch_path)

                nodule['patch_path'] = str(patch_path)
                manifest_data.append(nodule)

        except Exception as e:
            print(f"Skipping {patient_id} due to error: {e}")

    return pd.DataFrame(manifest_data)

print("Pipeline function successfully restored!")

Pipeline function successfully restored!


In [14]:
# Execute the preprocessing pipeline on ALL patients
# By setting max_scans=None, it will process all 1,010 subjects in your raw_data folder
print(f"Executing full dataset pipeline from: {RAW_DATA_DIR}")

# Run the full extraction
manifest_df = preprocess_full_dataset(RAW_DATA_DIR, OUTPUT_DIR, max_scans=None)

# Save the final full-scale manifest to your Google Drive
manifest_path = OUTPUT_DIR / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"\n✅ Full dataset preprocessing complete! Manifest saved to: {manifest_path}")
print(f"Total valid patches generated across all 1,010 patients: {len(manifest_df)}")

Executing full dataset pipeline from: /content/drive/MyDrive/Lung_Nodule_Project/raw_data


Preprocessing Patients:   0%|          | 0/1018 [00:00<?, ?it/s]

Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Skipping LIDC-IDRI-0874 due to error: No DICOM files found in: /content/drive/MyDrive/Lung_Nodule_Project/raw_data/LIDC-IDRI-0874/1.3.6.1.4.1.14519.5.2.1.6279.6

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/Lung_Nodule_Project/processed_patches'

In [15]:
import os
from google.colab import drive

# 1. Force remount Google Drive to wake it up after the 3.5-hour run
drive.mount('/content/drive', force_remount=True)

# 2. Explicitly define the save path and guarantee the folder exists
safe_output_dir = "/content/drive/MyDrive/Lung_Nodule_Project/processed_patches"
os.makedirs(safe_output_dir, exist_ok=True)

# 3. Save the manifest_df that is still safely in Python's memory!
safe_manifest_path = os.path.join(safe_output_dir, "manifest.csv")
manifest_df.to_csv(safe_manifest_path, index=False)

print(f"\n✅ CURED! Manifest successfully saved to: {safe_manifest_path}")
print(f"Total nodules ready for Phase 4 training: {len(manifest_df)}")

Mounted at /content/drive

✅ CURED! Manifest successfully saved to: /content/drive/MyDrive/Lung_Nodule_Project/processed_patches/manifest.csv
Total nodules ready for Phase 4 training: 1607
